# Campus Budget Coach

A student spending and savings assistant.




## Step 1 — Understand the problem

**User:** a university student who has a transaction export and wants to understand spending while saving for a specific goal.

**Business problem:** small purchases are hard to track alongside large bills. The student needs a clear spending summary and a realistic weekly savings amount, rather than a generic answer that ignores their transactions.

**Proposed result:** upload a CSV, inspect money in, money out and category totals, calculate a savings plan, then discuss the actual results with a friendly finance assistant. Success means that changing the file changes the summary and the context sent to Gemini, and changing a goal changes the weekly amount.

The assistant provides education about budgeting and saving. It asks a clarifying question when information is missing and redirects unrelated questions to its finance purpose.


## Step 2 — Identify inputs and outputs

| Input | Meaning and validation |
|---|---|
| CSV with `Date`, `Description`, `Category`, `Amount` | One row per transaction; dates in YYYY-MM-DD format; positive amounts are money in, negative amounts are money out. Numbers may include `$`. Blank descriptions or categories receive clear labels. Invalid amounts or dates stop the upload with a useful message. |
| Goal amount and amount already saved | Non-negative dollar values with at most two decimal places; goal must be greater than zero. |
| Weeks available | A positive whole number. |
| Weekly amount available | A non-negative dollar value chosen by the user. |
| Chat message | A non-empty question about the loaded data or savings plan. |
| `GOOGLE_API_KEY` | Read from Colab Secrets or a local environment variable. Never written into the notebook. |

Outputs are a spending summary, a category table, a savings plan showing the required weekly amount and any shortfall, and real Gemini replies using the current summary and plan.

**Data limitation:** money in can include refunds or transfers. Net cash flow is money in minus money out for the uploaded period; it is not the person's account balance or guaranteed future savings. The app does not guess a monthly rate from an incomplete export.


## Step 3 — Work an example by hand

Example transactions: wages **+\$1,000**, rent **-\$400**, groceries **-\$50**, bus **-\$25**, and another grocery purchase **-\$25**.

- Money in = \$1,000.
- Money out = \$400 + \$50 + \$25 + \$25 = **\$500**.
- Net cash flow = \$1,000 - \$500 = **\$500**.
- Category spending = Housing \$400, Food \$75, Transport \$25.

For a **\$500** goal, **\$100** already saved, **4** weeks remaining and **\$80** available each week:

- Remaining = \$500 - \$100 = **\$400**.
- Required each week = \$400 / 4 = **\$100**.
- Weekly shortfall = \$100 - \$80 = **\$20**.
- Projected saved at the user's current contribution = \$100 + 4 × \$80 = **\$420**.

A one-cent amount spread over three weeks still requires one cent per week when contributions must be whole cents. The calculation rounds the required contribution upwards so it cannot promise a goal that is a cent short.


## Step 4 — Write pseudocode before Python

```text
WHEN the user uploads a file and clicks Analyse:
    require a non-empty CSV with the four named columns
    FOR each row:
        validate the date
        convert the amount to integer cents
        replace blank category or description with a clear label
    create totals and a category dictionary
    FOR each transaction:
        add positive amounts to money in
        add the size of negative amounts to money out and its category
    show the summary and sorted category totals
    replace the session's previous summary and clear its old chat
    IF anything is invalid: explain the problem and clear stale results

WHEN the user clicks Calculate savings plan:
    validate goal, saved amount, weeks and weekly contribution
    remaining = maximum of (goal minus saved) and zero
    required weekly cents = remaining divided by weeks, rounded up
    shortfall = maximum of (required minus available) and zero
    show the required contribution, shortfall and projected total
    replace the current goal plan and clear its old chat

WHEN the user sends a chat message:
    require a message and a successfully loaded summary
    read the API key privately
    build a request with the finance system instruction
    include recent conversation, the current summary and current goal plan
    call Gemini with requests and a timeout
    IF successful: add the real reply to chat history
    ELSE: show a useful status without pretending a model replied
```

The savings planner is a custom tool: its validation, remaining amount, ceiling division, shortfall and projection are written explicitly rather than delegated to a financial library.
